# DiCE in Python - with the example of GSE239282

Based on this [Paper](https://doi.org/10.1093/nar/gkaf609). Since the paper divides the DiCE workflow into 6 Steps according to roman numerals, this example workflow will also be divided in these steps to show the progress.

Example Dataset: [GSE239282](https://www.ncbi.nlm.nih.gov/geo/geo2r/?acc=GSE239282) from GEO
- Age-related cognitive disorders (ACD) vs. controls
- Effects of music on their brains
- Only select ones with Timepoint 2 to remove duplicate accessions
- 48 total accessions, out of which 21 are control and 27 are ACD patients


In [1]:
import pandas as pd
import requests
import os

## Phase I: construction of gene candidate pool by DEA

### Step I.1: Download raw data from the [GEO Page on NCBI](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE239282) 
- Get both the normalized corrected expression for ACD and the Controls (`GSE239282_Norm_corrected_exp_ACD.txt.gz` and `GSE239282_Norm_corrected_exp_CONTROLS.txt.gz`)
- Place them in the `data` folder

Unzip the files first and load them into Pandas Dataframes

In [2]:
acd_file = os.path.join("..", "data", "GSE239282_Norm_corrected_exp_ACD.txt.gz")

acd_df =  pd.read_csv(acd_file, compression="gzip", sep=" ")

print(acd_df.shape)
acd_df.head()

(36155, 32)


,22_P_1003_1_t,22_P_1003_2_t,22_P_1006_1_t,22_P_1006_2_t,22_P_1009_1_t,22_P_1009_2_t,22_P_1052_1_t,22_P_1052_2_t,22_P_1055_1_t,22_P_1055_2_t,...,22_P_1137_1_t,22_P_1137_2_t,22_P_1139_1_t,22_P_1139_2_t,22_P_1144_1_t,22_P_1144_2_t,22_P_1145_1_t,22_P_1145_2_t,22_P_1149_1_t,22_P_1149_2_t
ENSG00000186827,8.610525,8.504742,8.589072,8.526195,8.595409,8.519858,8.432430,8.682836,8.587906,8.527360,...,8.562600,8.552666,8.712604,8.402663,8.609995,8.505272,8.584278,8.530989,8.341124,8.774143
ENSG00000186891,7.828481,8.016035,7.945485,7.899031,8.053736,7.790780,7.963202,7.881314,7.945408,7.899108,...,7.928480,7.916036,8.097148,7.747368,8.014888,7.829628,8.086238,7.758278,7.720037,8.124479
ENSG00000160072,7.836021,8.299189,8.035940,8.099270,8.174804,7.960406,7.895770,8.239440,8.020948,8.114262,...,8.146242,7.988968,8.298583,7.836627,8.202834,7.932376,8.145979,7.989231,8.003293,8.131917
ENSG00000041988,8.849681,9.008522,9.072159,8.786044,8.866810,8.991393,8.923828,8.934375,8.902255,8.955948,...,9.017802,8.840402,8.983226,8.874977,8.888825,8.969379,9.016611,8.841592,8.708116,9.150087
ENSG00000260179,6.503095,6.302982,6.531647,6.274430,6.321415,6.484662,6.401950,6.404127,6.334870,6.471207,...,6.291718,6.514359,6.621007,6.185070,6.403039,6.403039,6.403039,6.403039,6.403039,6.403039


In [3]:
control_file = os.path.join("..", "data", "GSE239282_Norm_corrected_exp_CONTROLS.txt.gz")
control_df = pd.read_csv(control_file, compression="gzip", sep=" ")

print(control_df.shape)

control_df.head()

(35865, 28)


,22_P_1001_1_M,22_P_1001_2_M,22_P_1004_1_M,22_P_1004_2_M,22_P_1013_1_t,22_P_1013_2_t,22_P_1020_1_t,22_P_1020_2_t,22_P_1021_1_t,22_P_1021_2_t,...,22_P_1065_1_t,22_P_1065_2_t,22_P_1069_1_t,22_P_1069_2_t,22_P_1079_1_t,22_P_1079_2_t,22_P_1103_1_t,22_P_1103_2_t,22_P_1120_1_t,22_P_1120_2_t
ENSG00000186827,8.883350,8.662983,8.635358,8.910976,8.731457,8.814876,8.818804,8.727529,8.561679,8.984654,...,8.786965,8.759369,8.976178,8.570156,8.791217,8.755116,8.896566,8.649768,8.598246,8.948087
ENSG00000186891,7.939168,7.999012,7.840740,8.097440,7.996373,7.941807,8.025952,7.912229,7.911350,8.026830,...,8.115303,7.822877,8.056440,7.881741,7.968946,7.969235,8.127619,7.810561,7.953573,7.984608
ENSG00000160072,8.389852,8.293670,8.049834,8.633688,8.372360,8.311163,8.481011,8.202511,8.040505,8.643017,...,8.461966,8.221556,8.325660,8.357862,8.445201,8.238321,8.504728,8.178794,8.169602,8.513920
ENSG00000041988,8.992454,9.223743,9.076469,9.139728,9.123871,9.092325,9.035888,9.180309,9.211157,9.005039,...,9.150607,9.065590,9.123860,9.092337,9.150915,9.065282,9.209681,9.006515,8.992529,9.223668
ENSG00000260179,6.477273,6.287379,6.382326,6.382326,6.501114,6.263538,6.489025,6.275627,6.382326,6.382326,...,6.453567,6.311084,6.382326,6.382326,6.478669,6.285982,6.382326,6.382326,6.521689,6.242963


Curious thing about this dataset: Sometimes, accessions are double - once at timepoint 1 (before treatment with music), and once at timepoint 2 (after treatment with music)

To account for this and make sure we don't count single individuals twice, we have to remove all patients/samples that have "_1_M" or "_1_t" in their name. 

This should halve the number of rows since there is always one measurement at timepoint 1 and one at timepoint 2.

In [4]:
print(f"ACD Columns before filtering: {len(acd_df.columns)} Columns, all columns: {list(acd_df.columns)}")
acd_timepoint2_columns = [column for column in acd_df if not "_1_t" in str(column) and not "_1_M" in str(column)]
print(f"ACD Columns after filtering: {len(acd_timepoint2_columns)} Columns, all columns: {acd_timepoint2_columns}")

print(f"Control Columns before filtering: {len(control_df.columns)} Columns, all columns: {list(control_df.columns)}")
control_timepoint2_columns = [column for column in control_df if not "_1_t" in str(column) and not "_1_M" in str(column)]
print(f"Control Columns after filtering: {len(control_timepoint2_columns)} Columns, all columns: {control_timepoint2_columns}")


ACD Columns before filtering: 32 Columns, all columns: ['22_P_1003_1_t', '22_P_1003_2_t', '22_P_1006_1_t', '22_P_1006_2_t', '22_P_1009_1_t', '22_P_1009_2_t', '22_P_1052_1_t', '22_P_1052_2_t', '22_P_1055_1_t', '22_P_1055_2_t', '22_P_1066_1_t', '22_P_1066_2_t', '22_P_1081_1_t', '22_P_1081_2_t', '22_P_1111_1_t', '22_P_1111_2_t', '22_P_1119_1_t', '22_P_1119_2_t', '22_P_1132_1_t', '22_P_1132_2_t', '22_P_1134_1_t', '22_P_1134_2_t', '22_P_1137_1_t', '22_P_1137_2_t', '22_P_1139_1_t', '22_P_1139_2_t', '22_P_1144_1_t', '22_P_1144_2_t', '22_P_1145_1_t', '22_P_1145_2_t', '22_P_1149_1_t', '22_P_1149_2_t']
ACD Columns after filtering: 16 Columns, all columns: ['22_P_1003_2_t', '22_P_1006_2_t', '22_P_1009_2_t', '22_P_1052_2_t', '22_P_1055_2_t', '22_P_1066_2_t', '22_P_1081_2_t', '22_P_1111_2_t', '22_P_1119_2_t', '22_P_1132_2_t', '22_P_1134_2_t', '22_P_1137_2_t', '22_P_1139_2_t', '22_P_1144_2_t', '22_P_1145_2_t', '22_P_1149_2_t']
Control Columns before filtering: 28 Columns, all columns: ['22_P_1001_1_

In [5]:
acd_df = acd_df[acd_timepoint2_columns]
print(acd_df.shape)
acd_df.head()

(36155, 16)


,22_P_1003_2_t,22_P_1006_2_t,22_P_1009_2_t,22_P_1052_2_t,22_P_1055_2_t,22_P_1066_2_t,22_P_1081_2_t,22_P_1111_2_t,22_P_1119_2_t,22_P_1132_2_t,22_P_1134_2_t,22_P_1137_2_t,22_P_1139_2_t,22_P_1144_2_t,22_P_1145_2_t,22_P_1149_2_t
ENSG00000186827,8.504742,8.526195,8.519858,8.682836,8.527360,8.574406,8.542985,8.331633,8.551066,8.468595,8.614118,8.552666,8.402663,8.505272,8.530989,8.774143
ENSG00000186891,8.016035,7.899031,7.790780,7.881314,7.899108,7.732916,7.749307,7.828268,7.922400,7.648196,8.019141,7.916036,7.747368,7.829628,7.758278,8.124479
ENSG00000160072,8.299189,8.099270,7.960406,8.239440,8.114262,7.934136,8.009329,7.869653,7.873457,7.917407,8.126334,7.988968,7.836627,7.932376,7.989231,8.131917
ENSG00000041988,9.008522,8.786044,8.991393,8.934375,8.955948,8.855553,8.882743,8.890947,8.949587,8.801069,8.966777,8.840402,8.874977,8.969379,8.841592,9.150087
ENSG00000260179,6.302982,6.274430,6.484662,6.404127,6.471207,6.403039,6.403039,6.475797,6.403039,6.403039,6.403039,6.514359,6.185070,6.403039,6.403039,6.403039


In [6]:
control_df = control_df[control_timepoint2_columns]
print(control_df.shape)
control_df.head()

(35865, 14)


,22_P_1001_2_M,22_P_1004_2_M,22_P_1013_2_t,22_P_1020_2_t,22_P_1021_2_t,22_P_1022_2_t,22_P_1056_2_t,22_P_1060_2_t,22_P_1062_2_t,22_P_1065_2_t,22_P_1069_2_t,22_P_1079_2_t,22_P_1103_2_t,22_P_1120_2_t
ENSG00000186827,8.662983,8.910976,8.814876,8.727529,8.984654,8.917620,8.792564,8.980667,8.785049,8.759369,8.570156,8.755116,8.649768,8.948087
ENSG00000186891,7.999012,8.097440,7.941807,7.912229,8.026830,8.023756,7.955509,8.093469,7.964896,7.822877,7.881741,7.969235,7.810561,7.984608
ENSG00000160072,8.293670,8.633688,8.311163,8.202511,8.643017,8.488877,8.382109,8.386469,8.233644,8.221556,8.357862,8.238321,8.178794,8.513920
ENSG00000041988,9.223743,9.139728,9.092325,9.180309,9.005039,9.165657,9.113144,9.118238,8.993156,9.065590,9.092337,9.065282,9.006515,9.223668
ENSG00000260179,6.287379,6.382326,6.263538,6.275627,6.382326,6.186864,6.230492,6.382326,6.382326,6.311084,6.382326,6.285982,6.382326,6.242963


Combine the two Dataframes into one

In [17]:
raw_data_df = pd.concat(objs=[acd_df, control_df], axis=1)
print(raw_data_df.shape)
raw_data_df.head()

(38327, 30)


,22_P_1003_2_t,22_P_1006_2_t,22_P_1009_2_t,22_P_1052_2_t,22_P_1055_2_t,22_P_1066_2_t,22_P_1081_2_t,22_P_1111_2_t,22_P_1119_2_t,22_P_1132_2_t,...,22_P_1021_2_t,22_P_1022_2_t,22_P_1056_2_t,22_P_1060_2_t,22_P_1062_2_t,22_P_1065_2_t,22_P_1069_2_t,22_P_1079_2_t,22_P_1103_2_t,22_P_1120_2_t
ENSG00000186827,8.504742,8.526195,8.519858,8.682836,8.527360,8.574406,8.542985,8.331633,8.551066,8.468595,...,8.984654,8.917620,8.792564,8.980667,8.785049,8.759369,8.570156,8.755116,8.649768,8.948087
ENSG00000186891,8.016035,7.899031,7.790780,7.881314,7.899108,7.732916,7.749307,7.828268,7.922400,7.648196,...,8.026830,8.023756,7.955509,8.093469,7.964896,7.822877,7.881741,7.969235,7.810561,7.984608
ENSG00000160072,8.299189,8.099270,7.960406,8.239440,8.114262,7.934136,8.009329,7.869653,7.873457,7.917407,...,8.643017,8.488877,8.382109,8.386469,8.233644,8.221556,8.357862,8.238321,8.178794,8.513920
ENSG00000041988,9.008522,8.786044,8.991393,8.934375,8.955948,8.855553,8.882743,8.890947,8.949587,8.801069,...,9.005039,9.165657,9.113144,9.118238,8.993156,9.065590,9.092337,9.065282,9.006515,9.223668
ENSG00000260179,6.302982,6.274430,6.484662,6.404127,6.471207,6.403039,6.403039,6.475797,6.403039,6.403039,...,6.382326,6.186864,6.230492,6.382326,6.382326,6.311084,6.382326,6.285982,6.382326,6.242963


### Step I.2: Perform DEA via GEO2R and filter for the resulting genes

The first part of the DiCE workflow, the construction of a gene candidate pool by DEA (Differential Expression Analysis) is done by GEO's GEO2R tool, available on the NCBI website.

- Go to the [GEO2R Page](https://www.ncbi.nlm.nih.gov/geo/geo2r/?acc=GSE239282) of this dataset
- Define two Groups based on the "Group" column: Control and ACD
- Assign all Control samples to the Control group and all ACD samples to the ACD group
  - You can already filter for timepoint 2 samples here, but it shouldn't make much of a difference overall
- (Optional for visualization) Go to Options below and set Significance level cut-off to 0
- Click "Analyze" or "Reanalyze" to run the GEO2R analysis
- Download the resulting table and put it into the `data` folder

We can then load the resulting table into a Pandas Dataframe

In [8]:
dea_file = os.path.join("..", "data", "GSE239282.top.table.tsv")
dea_results = pd.read_csv(filepath_or_buffer=dea_file, sep="\t")
print(dea_results.shape)
dea_results.head()

(16951, 9)


,GeneID,padj,pvalue,lfcSE,stat,log2FoldChange,baseMean,Symbol,Description
0,57124,5.730000e-14,3.380000e-18,0.3454,-8.697920,-3.004641,36.9,CD248,CD248 molecule
1,65982,1.060000e-13,1.250000e-17,0.1150,-8.547814,-0.982718,289.0,ZSCAN18,zinc finger and SCAN domain containing 18
2,107985900,3.560000e-13,6.300000e-17,0.2731,-8.359393,-2.283153,31.2,LOC107985900,uncharacterized LOC107985900
3,3983,4.600000e-12,1.090000e-15,0.1197,-8.016680,-0.959828,1180.0,ABLIM1,actin binding LIM protein 1
4,107985055,9.270000e-12,2.730000e-15,0.2245,7.902486,1.774105,23.1,LOC107985055,uncharacterized LOC107985055


Right now, this table is sorted by `padj`, the adjusted p-value. We want to only include those with a p-value of < 0.05 and, optionally, with a `log2FoldChange` of > 0.5.

In [ ]:
from python_dice.preparation import filter_dea_table

dea_filtered_results = filter_dea_table(dea_results)

Now, we filter the entire DF to only keep genes that are mentioned in the filtered DEA table

Since this is a "weird" database, the gene IDs of the raw data and the GEO2R analysis results do NOT match. Instead, the raw data has Ensembl IDs (like `ENSG00000174807`), while the GEO2R DEA results have the NCBI gene ID (e.g. `57124`)

To map these names, we will download a mapping file from HGNC between HGNC ID, Ensembl ID and Ncbi Gene ID using the [custom downloads](https://www.genenames.org/download/custom/) page.

Below, we download the file if it doesn't exist yet.

In [10]:
from python_dice.preparation import get_gene_id_mappings

gene_id_mappings = get_gene_id_mappings(force_download=False)

File ..\data\gene_id_mappings.txt already exists. If you want to download it again, set force_download to True.


Now, we map all of the NCBI gene IDs and all of the Ensembl gene IDs to HGNC IDs - this is the standard for all human genes and is the most easily convertible and comparable afterwards

As input, we give a single column (or in the case of the raw data, the index of the DataFrame) that will be a Pandas Series object. We also give it the parameter on which source gene IDs it will map from (here NCBI and Ensembl).

In [11]:
from python_dice.preparation import convert_series_to_hgnc

In [12]:
dea_filtered_results["GeneID"].head()

0        57124
1        65982
2    107985900
3         3983
4    107985055
Name: GeneID, dtype: int64

In [13]:
dea_filtered_results["GeneID"] = convert_series_to_hgnc(series=dea_filtered_results["GeneID"], gene_id_mappings=gene_id_mappings)

In [14]:
dea_filtered_results["GeneID"].head()

0    HGNC:18219
1    HGNC:21037
2           NaN
3       HGNC:78
4           NaN
Name: GeneID, dtype: str

We then remove all rows in the GeneID where there is a NaN

In [15]:
dea_filtered_results = dea_filtered_results[dea_filtered_results['GeneID'].notna()]   
dea_filtered_results.head()

,GeneID,padj,pvalue,lfcSE,stat,log2FoldChange,baseMean,Symbol,Description
0,HGNC:18219,5.730000e-14,3.380000e-18,0.3454,-8.697920,-3.004641,36.9,CD248,CD248 molecule
1,HGNC:21037,1.060000e-13,1.250000e-17,0.1150,-8.547814,-0.982718,289.0,ZSCAN18,zinc finger and SCAN domain containing 18
3,HGNC:78,4.600000e-12,1.090000e-15,0.1197,-8.016680,-0.959828,1180.0,ABLIM1,actin binding LIM protein 1
5,HGNC:10630,1.020000e-11,3.600000e-15,0.1367,7.868078,1.075797,1300.0,CCL4,C-C motif chemokine ligand 4
6,HGNC:9861,1.380000e-11,5.690000e-15,0.1229,7.810563,0.959738,401.0,RAP2A,"RAP2A, member of RAS oncogene family"


Now we will also do the same for the raw data

In [18]:
raw_data_df.head()

,22_P_1003_2_t,22_P_1006_2_t,22_P_1009_2_t,22_P_1052_2_t,22_P_1055_2_t,22_P_1066_2_t,22_P_1081_2_t,22_P_1111_2_t,22_P_1119_2_t,22_P_1132_2_t,...,22_P_1021_2_t,22_P_1022_2_t,22_P_1056_2_t,22_P_1060_2_t,22_P_1062_2_t,22_P_1065_2_t,22_P_1069_2_t,22_P_1079_2_t,22_P_1103_2_t,22_P_1120_2_t
ENSG00000186827,8.504742,8.526195,8.519858,8.682836,8.527360,8.574406,8.542985,8.331633,8.551066,8.468595,...,8.984654,8.917620,8.792564,8.980667,8.785049,8.759369,8.570156,8.755116,8.649768,8.948087
ENSG00000186891,8.016035,7.899031,7.790780,7.881314,7.899108,7.732916,7.749307,7.828268,7.922400,7.648196,...,8.026830,8.023756,7.955509,8.093469,7.964896,7.822877,7.881741,7.969235,7.810561,7.984608
ENSG00000160072,8.299189,8.099270,7.960406,8.239440,8.114262,7.934136,8.009329,7.869653,7.873457,7.917407,...,8.643017,8.488877,8.382109,8.386469,8.233644,8.221556,8.357862,8.238321,8.178794,8.513920
ENSG00000041988,9.008522,8.786044,8.991393,8.934375,8.955948,8.855553,8.882743,8.890947,8.949587,8.801069,...,9.005039,9.165657,9.113144,9.118238,8.993156,9.065590,9.092337,9.065282,9.006515,9.223668
ENSG00000260179,6.302982,6.274430,6.484662,6.404127,6.471207,6.403039,6.403039,6.475797,6.403039,6.403039,...,6.382326,6.186864,6.230492,6.382326,6.382326,6.311084,6.382326,6.285982,6.382326,6.242963


In [21]:
raw_data_df.index = convert_series_to_hgnc(series=raw_data_df.index, gene_id_mappings=gene_id_mappings)
raw_data_df = raw_data_df[raw_data_df.index.notna()]

In [22]:
raw_data_df.head()

,22_P_1003_2_t,22_P_1006_2_t,22_P_1009_2_t,22_P_1052_2_t,22_P_1055_2_t,22_P_1066_2_t,22_P_1081_2_t,22_P_1111_2_t,22_P_1119_2_t,22_P_1132_2_t,...,22_P_1021_2_t,22_P_1022_2_t,22_P_1056_2_t,22_P_1060_2_t,22_P_1062_2_t,22_P_1065_2_t,22_P_1069_2_t,22_P_1079_2_t,22_P_1103_2_t,22_P_1120_2_t
HGNC:11918,8.504742,8.526195,8.519858,8.682836,8.527360,8.574406,8.542985,8.331633,8.551066,8.468595,...,8.984654,8.917620,8.792564,8.980667,8.785049,8.759369,8.570156,8.755116,8.649768,8.948087
HGNC:11914,8.016035,7.899031,7.790780,7.881314,7.899108,7.732916,7.749307,7.828268,7.922400,7.648196,...,8.026830,8.023756,7.955509,8.093469,7.964896,7.822877,7.881741,7.969235,7.810561,7.984608
HGNC:24007,8.299189,8.099270,7.960406,8.239440,8.114262,7.934136,8.009329,7.869653,7.873457,7.917407,...,8.643017,8.488877,8.382109,8.386469,8.233644,8.221556,8.357862,8.238321,8.178794,8.513920
HGNC:20855,9.008522,8.786044,8.991393,8.934375,8.955948,8.855553,8.882743,8.890947,8.949587,8.801069,...,9.005039,9.165657,9.113144,9.118238,8.993156,9.065590,9.092337,9.065282,9.006515,9.223668
HGNC:42092,6.338601,6.333603,6.554030,6.338317,6.438658,6.306428,6.507486,6.541511,6.533949,6.510794,...,6.708786,6.604246,6.620081,6.614662,6.620081,6.620081,6.620081,6.696400,6.686163,6.487814


TODO: Now we simply filter the raw data by filtering all rows and only keeping the ones with indexes that are also present in the DEA result's Gene ID column

If the data is not normalised yet, you will have to normalize to z-scores first - in this case, the data is already normalised.

## Phase II: selection of most discriminative genes using IG

Information Gain (IG) can be calculated using `scikit-learn`'s implementation of Mutual Information in its method [mutual_info_classif](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.mutual_info_classif.html)

For this method, the data has to be rows for samples and columns for features (in our case genes)

In [16]:
transposed_full_df = full_df.T 

transposed_full_df.head()

,ENSG00000186827,ENSG00000186891,ENSG00000160072,ENSG00000041988,ENSG00000260179,ENSG00000234396,ENSG00000225972,ENSG00000224315,ENSG00000198744,ENSG00000279928,...,ENSG00000275523,ENSG00000227702,ENSG00000225637,ENSG00000142182,ENSG00000187766,ENSG00000229880,ENSG00000185437,ENSG00000210140,ENSG00000277761,ENSG00000273730
22_P_1003_2_t,8.504742,8.016035,8.299189,9.008522,6.302982,6.354442,6.338601,6.377931,6.373415,6.794537,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
22_P_1006_2_t,8.526195,7.899031,8.099270,8.786044,6.274430,6.280124,6.333603,6.505103,6.481892,6.908477,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
22_P_1009_2_t,8.519858,7.790780,7.960406,8.991393,6.484662,6.354442,6.554030,6.403731,6.368179,7.092747,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
22_P_1052_2_t,8.682836,7.881314,8.239440,8.934375,6.404127,6.354442,6.338317,6.403731,6.486531,6.839125,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
22_P_1055_2_t,8.527360,7.899108,8.114262,8.955948,6.471207,6.354442,6.438658,6.403731,6.501887,7.018676,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
